# Enhanced Routing — Waypoints, Steps, Alternatives & More

Demonstrates the advanced OSRM routing features exposed by `route()`:

| Feature              | Parameter               | Description                              |
| -------------------- | ----------------------- | ---------------------------------------- |
| Waypoints            | `waypoints`             | Route through intermediate points        |
| Turn-by-turn steps   | `steps=True`            | Manoeuvre instructions per leg           |
| Alternative routes   | `alternatives=True`     | Request multiple route options           |
| Annotations          | `annotations=[...]`     | Per-segment duration/distance/speed data |
| Exclude road classes | `exclude=[...]`         | Avoid motorway, toll, ferry, etc.        |
| Geometry format      | `geometries="geojson"`  | GeoJSON, polyline, or polyline6          |
| Overview detail      | `overview="simplified"` | Full, simplified, or no geometry         |

For basic A-to-B routing across all three profiles, see
[02-routing.ipynb](02-routing.ipynb).

## Prerequisites

| Service        | Default port | Role                        |
| -------------- | ------------ | --------------------------- |
| OSRM + HAProxy | 80           | Road routing (all profiles) |
| Nominatim      | 8080         | Used to geocode locations   |
| Photon         | 2322         | Enriches geocoding results  |


______________________________________________________________________

## Start Services (optional)

Run the cell below to start the combined stack and wait until routing and geocoding
services respond. **Skip if the services are already running.**


In [ ]:
import pathlib
import subprocess
import time

import requests as _requests

from airgap_geo.settings import NOMINATIM_URL, OSRM_API, PHOTON_API


def _find_repo_root(start: pathlib.Path) -> pathlib.Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


_REPO_ROOT = _find_repo_root(pathlib.Path().resolve())
_COMPOSE_FILE = _REPO_ROOT / "docker" / "docker-compose.yml"
_ENV_FILE = _REPO_ROOT / ".env"

_HEALTH_TIMEOUT = 120

_ENDPOINTS = {
    "Nominatim": NOMINATIM_URL,
    "Photon": PHOTON_API,
    "OSRM / HAProxy": OSRM_API,
}


def start_services(timeout: int = _HEALTH_TIMEOUT) -> None:  # noqa: D103
    print(f"Starting stack from {_COMPOSE_FILE.relative_to(_REPO_ROOT)} ...")
    proc = subprocess.run(
        [
            "docker",
            "compose",
            "-f",
            str(_COMPOSE_FILE),
            "--env-file",
            str(_ENV_FILE),
            "up",
            "-d",
        ],
        capture_output=True,
        text=True,
    )
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())

    ready = {name: False for name in _ENDPOINTS}
    deadline = time.monotonic() + timeout
    print(f"\nPolling services (timeout {timeout}s) ...")

    while time.monotonic() < deadline:
        for name, url in _ENDPOINTS.items():
            if ready[name]:
                continue
            try:
                _requests.get(url, timeout=3)
                ready[name] = True
                print(f"  \u2713  {name} is up  ({url})")
            except Exception:
                pass
        if all(ready.values()):
            break
        time.sleep(3)

    still_down = [n for n, ok in ready.items() if not ok]
    if still_down:
        print(f"\n[WARNING] Timed out waiting for: {', '.join(still_down)}")
    else:
        print("\nAll services are up and ready.")


start_services()

## Setup — Imports and Configuration


In [ ]:
import importlib

import folium
import httpx
import pandas as pd
import requests

import airgap_geo.settings as _settings

importlib.reload(_settings)

from airgap_geo import geocoder, route  # noqa: E402
from airgap_geo.settings import OSRM_API  # noqa: E402

client = httpx.AsyncClient()

print(f"OSRM / HAProxy : {OSRM_API}")


def _decode_polyline(encoded: str) -> list[tuple[float, float]]:
    """Decode an OSRM / Google encoded polyline string into (lat, lon) pairs."""
    coords: list[tuple[float, float]] = []
    index = 0
    lat = 0
    lng = 0
    while index < len(encoded):
        for is_lng in (False, True):
            shift, result = 0, 0
            while True:
                b = ord(encoded[index]) - 63
                index += 1
                result |= (b & 0x1F) << shift
                shift += 5
                if b < 0x20:
                    break
            delta = ~(result >> 1) if (result & 1) else (result >> 1)
            if is_lng:
                lng += delta
            else:
                lat += delta
        coords.append((lat / 1e5, lng / 1e5))
    return coords

## Health Check


In [ ]:
try:
    r = requests.get(OSRM_API, timeout=5)
    print(f"OSRM / HAProxy  \u2713  HTTP {r.status_code}  ({OSRM_API})")
except Exception:
    print(f"OSRM / HAProxy  \u2717  unreachable  ({OSRM_API})")

______________________________________________________________________

## 1 — Waypoints (multi-leg routing)

Route through an intermediate waypoint: King's Cross \\u2192 Bank of England \\u2192 Victoria.
OSRM returns one route per alternative, each broken into **legs** \\u2014 one leg per
consecutive waypoint pair. The `waypoints` field on the result records the input
intermediate points; `snapped_waypoints` records where OSRM actually placed them
on the road network.


In [ ]:
result_kx = await geocoder("King's Cross Station, London", client)
result_bank = await geocoder("Bank of England, London", client)
result_vic = await geocoder("Victoria Station, London", client)

kx_coords = (result_kx["geo"]["lat"], result_kx["geo"]["lon"])
bank_coords = (result_bank["geo"]["lat"], result_bank["geo"]["lon"])
vic_coords = (result_vic["geo"]["lat"], result_vic["geo"]["lon"])

print(f"King's Cross    : {kx_coords}")
print(f"Bank of England : {bank_coords}")
print(f"Victoria        : {vic_coords}")

r_wp = await route(
    kx_coords, vic_coords, client, profile="driving", waypoints=[bank_coords]
)

print(f"\nTotal distance : {r_wp.routes[0].distance_m / 1000:.2f} km")
print(f"Total duration : {r_wp.routes[0].duration_s / 60:.1f} min")
print(f"Legs           : {len(r_wp.routes[0].legs)}")
print(f"Input waypoints  : {[(wp.lat, wp.lon) for wp in r_wp.waypoints]}")
print(f"Snapped waypoints: {[(wp.lat, wp.lon) for wp in r_wp.snapped_waypoints]}")

In [ ]:
# Per-leg summary table
leg_rows = []
stop_labels = ["King's Cross", "Bank of England", "Victoria"]
for i, leg in enumerate(r_wp.routes[0].legs):
    leg_rows.append(
        {
            "Leg": f"{stop_labels[i]} \u2192 {stop_labels[i + 1]}",
            "Distance (km)": round(leg.distance_m / 1000, 2),
            "Duration (min)": round(leg.duration_s / 60, 1),
        }
    )
pd.DataFrame(leg_rows).set_index("Leg")

In [ ]:
# Map: three stops and the multi-leg driving route
all_lats = [kx_coords[0], bank_coords[0], vic_coords[0]]
all_lons = [kx_coords[1], bank_coords[1], vic_coords[1]]
mid_lat_wp = sum(all_lats) / 3
mid_lon_wp = sum(all_lons) / 3

m_wp = folium.Map(location=[mid_lat_wp, mid_lon_wp], zoom_start=13)

stop_icons = [
    (kx_coords, "King's Cross", "green", "play"),
    (bank_coords, "Bank of England", "orange", "map-marker"),
    (vic_coords, "Victoria", "red", "stop"),
]
for coords, label, colour, icon_name in stop_icons:
    folium.Marker(
        list(coords),
        tooltip=label,
        popup=folium.Popup(label, max_width=200),
        icon=folium.Icon(color=colour, icon=icon_name),
    ).add_to(m_wp)

encoded_full = r_wp.routes[0].geometry
if encoded_full:
    folium.PolyLine(
        _decode_polyline(encoded_full),
        color="royalblue",
        weight=4,
        tooltip="Driving route (via Bank of England)",
    ).add_to(m_wp)

m_wp

______________________________________________________________________

## 2 — Turn-by-turn Steps

Pass `steps=True` to get manoeuvre instructions for each leg. Each leg then contains
a list of `RouteTurnStep` objects with road name, distance, duration, and a
`RouteManoeuvre` describing the turn type (depart, turn, arrive, etc.) and modifier
(straight, left, right, etc.).


In [ ]:
r_steps = await route(kx_coords, vic_coords, client, profile="driving", steps=True)

leg = r_steps.routes[0].legs[0]
print(f"Steps in first leg: {len(leg.steps)}\n")

step_rows = []
for step in leg.steps:
    manoeuvre_type = step.manoeuvre.type if step.manoeuvre else "-"
    manoeuvre_mod = (
        step.manoeuvre.modifier if step.manoeuvre and step.manoeuvre.modifier else "-"
    )
    step_rows.append(
        {
            "Road name": step.name or "(unnamed)",
            "Manoeuvre": manoeuvre_type,
            "Modifier": manoeuvre_mod,
            "Distance (m)": round(step.distance_m, 0),
            "Duration (s)": round(step.duration_s, 0),
        }
    )

pd.DataFrame(step_rows)

______________________________________________________________________

## 3 — Alternative Routes

Pass `alternatives=True` (or an integer count) to request multiple route options.
OSRM returns them ranked by duration. Render all alternatives on the same map in
distinct colours so they can be compared visually.


In [ ]:
r_alt = await route(kx_coords, vic_coords, client, profile="driving", alternatives=3)

print(f"Routes returned: {len(r_alt.routes)}\n")

alt_rows = []
for i, r in enumerate(r_alt.routes):
    alt_rows.append(
        {
            "Route": f"Route {i + 1}",
            "Distance (km)": round(r.distance_m / 1000, 2),
            "Duration (min)": round(r.duration_s / 60, 1),
        }
    )

pd.DataFrame(alt_rows).set_index("Route")

In [ ]:
_ALT_COLOURS = ["royalblue", "crimson", "forestgreen", "darkorange"]

mid_alt = ((kx_coords[0] + vic_coords[0]) / 2, (kx_coords[1] + vic_coords[1]) / 2)
m_alt = folium.Map(location=list(mid_alt), zoom_start=13)

folium.Marker(
    list(kx_coords),
    tooltip="King's Cross (origin)",
    icon=folium.Icon(color="green", icon="play"),
).add_to(m_alt)
folium.Marker(
    list(vic_coords),
    tooltip="Victoria (destination)",
    icon=folium.Icon(color="red", icon="stop"),
).add_to(m_alt)

for i, r in enumerate(r_alt.routes):
    encoded = r.geometry
    if encoded:
        colour = _ALT_COLOURS[i % len(_ALT_COLOURS)]
        folium.PolyLine(
            _decode_polyline(encoded),
            color=colour,
            weight=4,
            opacity=0.8,
            tooltip=f"Route {i + 1}: {r.distance_m / 1000:.2f} km, {r.duration_s / 60:.1f} min",
        ).add_to(m_alt)

m_alt

______________________________________________________________________

## 4 — Per-segment Annotations

Pass `annotations=["duration", "distance"]` (and optionally `"speed"`, `"nodes"`,
`"weight"`, `"datasources"`) to receive per-road-segment data on each leg. This is
useful for travel-time analysis, isochrones, or heatmaps.

The annotation data is available as `leg.annotation` on each `RouteLeg`.


In [ ]:
r_ann = await route(
    kx_coords,
    vic_coords,
    client,
    profile="driving",
    annotations=["duration", "distance"],
)

leg = r_ann.routes[0].legs[0]
ann = leg.annotation or {}

durations = ann.get("duration", [])
distances = ann.get("distance", [])

print(f"Segments in first leg: {len(durations)}")
print(
    f"Total annotated duration : {sum(durations):.1f}s  (leg summary: {leg.duration_s:.1f}s)"
)
print(
    f"Total annotated distance : {sum(distances):.1f}m  (leg summary: {leg.distance_m:.1f}m)\n"
)

sample_rows = [
    {"Segment": i + 1, "Duration (s)": round(d, 2), "Distance (m)": round(dist, 1)}
    for i, (d, dist) in enumerate(zip(durations[:10], distances[:10]))  # noqa: B905
]
pd.DataFrame(sample_rows).set_index("Segment")

______________________________________________________________________

## 5 — Exclude Road Classes

The `exclude` parameter tells OSRM to avoid specific road classes (e.g. `"motorway"`,
`"toll"`, `"ferry"`). Available classes depend on the Lua profile used during data
preparation. Compare a route with and without motorway exclusion to see the effect on
distance and duration.


In [ ]:
result_bham = await geocoder("Birmingham New Street Station", client)
bham_coords = (result_bham["geo"]["lat"], result_bham["geo"]["lon"])

r_default = await route(kx_coords, bham_coords, client, profile="driving")
r_no_motorway = await route(
    kx_coords, bham_coords, client, profile="driving", exclude=["motorway"]
)

rows_excl = []
for label, r in [
    ("Default (with motorway)", r_default),
    ("Exclude motorway", r_no_motorway),
]:
    if r and r.routes:
        rows_excl.append(
            {
                "Variant": label,
                "Distance (km)": round(r.routes[0].distance_m / 1000, 2),
                "Duration (min)": round(r.routes[0].duration_s / 60, 1),
            }
        )
    else:
        rows_excl.append(
            {"Variant": label, "Distance (km)": "N/A", "Duration (min)": "N/A"}
        )

pd.DataFrame(rows_excl).set_index("Variant")

In [ ]:
mid_excl = (
    (kx_coords[0] + bham_coords[0]) / 2,
    (kx_coords[1] + bham_coords[1]) / 2,
)
m_excl = folium.Map(location=list(mid_excl), zoom_start=8)

folium.Marker(
    list(kx_coords),
    tooltip="London (King's Cross)",
    icon=folium.Icon(color="green", icon="play"),
).add_to(m_excl)
folium.Marker(
    list(bham_coords),
    tooltip="Birmingham New Street",
    icon=folium.Icon(color="red", icon="stop"),
).add_to(m_excl)

_excl_styles = [
    (r_default, "royalblue", "Default"),
    (r_no_motorway, "crimson", "No motorway"),
]
for r, colour, label in _excl_styles:
    if r and r.routes and r.routes[0].geometry:
        folium.PolyLine(
            _decode_polyline(r.routes[0].geometry),
            color=colour,
            weight=3,
            opacity=0.8,
            tooltip=f"{label}: {r.routes[0].distance_m / 1000:.0f} km, {r.routes[0].duration_s / 60:.0f} min",
        ).add_to(m_excl)

m_excl

______________________________________________________________________

## Teardown — Close HTTP Client


In [ ]:
await client.aclose()

______________________________________________________________________

## Stop Services (optional)

Run the cell below to stop and remove all containers. Persistent data volumes
are **not** removed.


In [ ]:
def stop_services() -> None:  # noqa: D103
    print(f"Stopping stack from {_COMPOSE_FILE.relative_to(_REPO_ROOT)} ...")
    proc = subprocess.run(
        [
            "docker",
            "compose",
            "-f",
            str(_COMPOSE_FILE),
            "--env-file",
            str(_ENV_FILE),
            "down",
        ],
        capture_output=True,
        text=True,
    )
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    print("Stack stopped.")


stop_services()